# 📊 Extracción de Características y KPIs en Yacimientos
## Módulo 2 · Feature Engineering & Dashboards Interactivos · Capacitación SLB

### Objetivos de la sesión:
1. Calcular KPIs físicos avanzados de ingeniería de yacimientos (Water-Cut, GOR, PI).
2. Manejar divisiones por cero provocadas por pozos en shut-in (cerrados).
3. Estudiar correlaciones físicas entre presiones y caudales usando mapas de calor.
4. Construir y desplegar un dashboard interactivo en vivo usando **Streamlit**.

## ¿Qué librerías cargaremos para la ingeniería de variables?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## ¿Cómo cargamos y limpiamos los datos iniciales?

In [ ]:
# Descargar el dataset directamente desde GitHub
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_2_eda/data/produccion_pozos.csv -O produccion_pozos.csv

df = pd.read_csv('produccion_pozos.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Limpieza básica rápida de transductores y spikes
df.loc[df['BHP_psi'] == -999.0, 'BHP_psi'] = np.nan
df['BHP_psi'] = df.groupby('Pozo')['BHP_psi'].transform(lambda x: x.interpolate(method='linear'))
df.loc[df['Qo_bpd'] > 10000, 'Qo_bpd'] = np.nan
df['Qo_bpd'] = df.groupby('Pozo')['Qo_bpd'].transform(lambda x: x.interpolate(method='linear'))
df.head()

# 1. Creación de KPIs Físicos

## ¿Cómo calculamos el Water-Cut (Corte de Agua) controlando las divisiones por cero en cierres de pozo?

In [ ]:
# TU CÓDIGO AQUÍ: Calcula el Water-Cut como Qw / (Qo + Qw) y rellena los resultantes NaN con 0
df['Water_Cut'] = None

df['Water_Cut'].describe()

## ¿Cómo calculamos la relación Gas-Petróleo (GOR) en miles de pies cúbicos por barril?

In [ ]:
# TU CÓDIGO AQUÍ: GOR = (Qg_mscfd * 1000) / Qo. Rellena los resultantes NaN/inf con 0
df['GOR'] = None

df['GOR'].describe()

## ¿Cómo calculamos el Índice de Productividad (PI) asumiendo una presión estática de yacimiento de 3300 psi?

In [ ]:
# TU CÓDIGO AQUÍ: PI = Qo / (3300 - BHP). Reemplaza divisiones por cero con 0
df['PI'] = None

df['PI'].describe()

# 2. Análisis de Correlaciones

## ¿Cómo visualizamos la correlación de Pearson mediante un Heatmap?

In [ ]:
# TU CÓDIGO AQUÍ: Calcula la matriz de correlación de las columnas numéricas y grafícala con sns.heatmap
corr = None

plt.title('Matriz de Correlación de Parámetros Operativos')
plt.show()

## ¿Qué relación física podemos deducir entre la BHP y el Caudal de crudo (Qo)?

In [ ]:
# TU CÓDIGO AQUÍ: Genera un gráfico de dispersión (scatterplot) entre BHP_psi y Qo_bpd


# 3. Dashboard Web con Streamlit

## ¿Cómo escribimos el archivo `app.py` que definirá nuestra interfaz interactiva?

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("🛢️ Dashboard de Producción de Pozos - SLB")
df = pd.read_csv('produccion_pozos.csv')

pozo = st.sidebar.selectbox("Selecciona el Pozo", df['Pozo'].unique())
df_pozo = df[df['Pozo'] == pozo]

st.metric("Caudal Promedio (bpd)", f"{df_pozo['Qo_bpd'].mean():.2f}")
fig = px.line(df_pozo, x='Fecha', y='Qo_bpd', title='Evolución de Producción')
st.plotly_chart(fig)

## ¿Cómo instalamos `localtunnel` de forma silenciosa para exponer el servidor de Colab a internet?

In [ ]:
!npm install -g localtunnel

## ¿Cómo ejecutamos la app de Streamlit en segundo plano y abrimos el túnel público?

In [ ]:
# Correr streamlit de fondo con protecciones de red desactivadas para localtunnel
import subprocess
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# Abrir el túnel local en segundo plano
get_ipython().system_raw('npx localtunnel --port 8501 &')